# LLM Evaluation Dashboard 3

`database/sql/restaurant.db` 기반 골드셋 생성, 평가 실행, 결과 시각화를 개발 환경에서 확인하기 위한 `src_test3` 전용 노트북입니다.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
from IPython.display import HTML, display

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'src_test3' else CURRENT_DIR
SRC_TEST_DIR = PROJECT_ROOT / 'src_test3'
DB_PATH = PROJECT_ROOT / 'database' / 'sql' / 'restaurant.db'
GOLDSET_PATH = SRC_TEST_DIR / 'llm_goldset.json'
JSON_REPORT_PATH = SRC_TEST_DIR / 'llm_eval_report.json'
HTML_REPORT_PATH = SRC_TEST_DIR / 'llm_eval_report.html'

print('CURRENT_DIR =', CURRENT_DIR)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('SRC_TEST_DIR =', SRC_TEST_DIR)
print('DB_PATH exists =', DB_PATH.exists())
print('OPENAI_API_KEY set =', bool(os.getenv('OPENAI_API_KEY')))


CURRENT_DIR = c:\MinHyeok\skn26_3rd_3rd\3rd_project\src_test3
PROJECT_ROOT = c:\MinHyeok\skn26_3rd_3rd\3rd_project
SRC_TEST_DIR = c:\MinHyeok\skn26_3rd_3rd\3rd_project\src_test3
DB_PATH exists = True
OPENAI_API_KEY set = True


## 1. 골드셋 재생성

In [2]:
result = subprocess.run(
    [sys.executable, str(SRC_TEST_DIR / 'build_llm_goldset.py')],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print('returncode =', result.returncode)


goldset_written: C:\MinHyeok\skn26_3rd_3rd\3rd_project\src_test3\llm_goldset.json
total_cases: 50
fixed_cases: 20
embedding_cases: 30

returncode = 0


## 2. 골드셋 샘플 확인

In [3]:
goldset = json.loads(GOLDSET_PATH.read_text(encoding='utf-8'))
print('total_cases =', len(goldset))
goldset[:3]


total_cases = 50


[{'case_id': 'fixed_001_hours',
  'source': 'restaurant.db',
  'query_type': 'fixed',
  'question': '유태우스시 영업시간 알려줘',
  'expected_route': 'fixed',
  'payload_checks': [{'keys': ['restaurant'], 'contains_any': ['유태우스시']}],
  'expected_targets': {'restaurant_codes': ['RES0000'],
   'restaurant_names': ['유태우스시']},
  'answer_checks': {'must_include_any': ['유태우스시', '영업', '시간'],
   'must_include_all': [],
   'must_not_include': []},
  'min_used_restaurants': 1,
  'metadata': {'restaurant_code': 'RES0000',
   'restaurant_name': '유태우스시',
   'menu_keyword': '회전초밥'}},
 {'case_id': 'fixed_001_menu',
  'source': 'restaurant.db',
  'query_type': 'fixed',
  'question': '유태우스시에 회전초밥 있어?',
  'expected_route': 'fixed',
  'payload_checks': [{'keys': ['restaurant'], 'contains_any': ['유태우스시']},
   {'keys': ['menu'], 'contains_any': ['회전초밥']}],
  'expected_targets': {'restaurant_codes': ['RES0000'],
   'restaurant_names': ['유태우스시']},
  'answer_checks': {'must_include_any': ['유태우스시', '회전초밥'],
   'must_inclu

## 3. 평가 실행

In [4]:
result = subprocess.run(
    [sys.executable, str(SRC_TEST_DIR / 'evaluate_llm.py')],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print('returncode =', result.returncode)


Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\MIN\miniconda3\envs\aistudy_env\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\MIN\miniconda3\envs\aistudy_env\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\MIN\miniconda3\envs\aistudy_env\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 63: illegal multibyte sequence


None
returncode = 1


## 4. 결과 로드

In [5]:
report = json.loads(JSON_REPORT_PATH.read_text(encoding='utf-8'))
report.keys()


dict_keys(['started_at', 'finished_at', 'goldset_path', 'preflight', 'summary', 'type_summary', 'environment_failure', 'cases'])

## 5. 전체 요약 표

In [6]:
summary = report['summary']
overview_rows = [
    ('cases', summary['total_cases']),
    ('passed', summary['passed_cases']),
    ('failed', summary['failed_cases']),
    ('pass_rate', f"{summary['pass_rate'] * 100:.1f}%"),
    ('avg_score', f"{summary['average_score']:.2f}"),
    ('route_acc', f"{summary['component_accuracy']['route'] * 100:.1f}%"),
    ('payload_acc', f"{summary['component_accuracy']['payload'] * 100:.1f}%"),
    ('target_acc', f"{summary['component_accuracy']['target'] * 100:.1f}%"),
    ('answer_acc', f"{summary['component_accuracy']['answer'] * 100:.1f}%"),
    ('retrieval_acc', f"{summary['component_accuracy']['retrieval'] * 100:.1f}%"),
]
html_rows = ''.join(
    f'<tr><td style="padding:8px;border:1px solid #ddd;">{k}</td><td style="padding:8px;border:1px solid #ddd;">{v}</td></tr>'
    for k, v in overview_rows
)
display(HTML(
    '<table style="border-collapse:collapse;min-width:360px;">'
    '<thead><tr><th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">metric</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">value</th></tr></thead>'
    f'<tbody>{html_rows}</tbody></table>'
))


metric,value
cases,50
passed,41
failed,9
pass_rate,82.0%
avg_score,0.97
route_acc,100.0%
payload_acc,86.0%
target_acc,96.0%
answer_acc,100.0%
retrieval_acc,100.0%


## 6. Query Type 요약

In [ ]:
type_rows = report.get('type_summary', [])
html_rows = ''.join(
    (
        '<tr>'
        f'<td style="padding:8px;border:1px solid #ddd;">{row["query_type"]}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{row["total"]}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{row["passed"]}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{row["failed"]}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{row["pass_rate"] * 100:.1f}%</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{row["average_score"]:.2f}</td>'
        '</tr>'
    )
    for row in type_rows
)
display(HTML(
    '<table style="border-collapse:collapse;min-width:640px;">'
    '<thead><tr>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">type</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">total</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">passed</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">failed</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">pass_rate</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">avg_score</th>'
    '</tr></thead>'
    f'<tbody>{html_rows}</tbody></table>'
))


## 7. 실패 케이스만 보기

In [ ]:
failed_cases = [item for item in report['cases'] if not item.get('passed')]
print('failed_cases =', len(failed_cases))
html_rows = ''.join(
    (
        '<tr>'
        f'<td style="padding:8px;border:1px solid #ddd;">{item["case_id"]}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{item["query_type"]}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{item["overall_score"]:.2f}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{item["failure_reason"]}</td>'
        f'<td style="padding:8px;border:1px solid #ddd;">{item["question"]}</td>'
        '</tr>'
    )
    for item in failed_cases
)
display(HTML(
    '<div style="max-height:520px;overflow:auto;border:1px solid #ddd;">'
    '<table style="border-collapse:collapse;min-width:1100px;width:100%;">'
    '<thead><tr>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">case_id</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">type</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">score</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">reason</th>'
    '<th style="padding:8px;border:1px solid #ddd;background:#f4efe8;">question</th>'
    '</tr></thead>'
    f'<tbody>{html_rows}</tbody></table></div>'
))


## 8. HTML 리포트 그대로 보기

In [ ]:
display(HTML(HTML_REPORT_PATH.read_text(encoding='utf-8')))
